# Q8 v2: Estendendo o Modelo Hierárquico com Features Harmônicas e Sub-gêneros

**Notebook:** `04_q8_features_extras.ipynb`  
**Foco:** incluir `key`, `mode`, `time_signature` e sub-gêneros (multi-hot) ao modelo hierárquico Bayesiano Q8.

**Sumário:**
- Cíclica para `key` (sin/cos em 12 classes de altura) preserva distância harmônica.
- `mode` mantido como binário nativo.
- `time_signature` em one-hot esparso com 4/4 como referência.
- `generos_top30` em matriz esparsa multi-hot (top-30 tokens mais frequentes).
- `tonalidade` e `tonalidade_completa` capturadas implicitamente pelos termos aditivos + cross-terms.
- Modelagem: Gaussian (NUTS via NumPyro em T4 GPU) + Bernoulli (ADVI mean-field).
- Sensitivity drop-one para quantificar contribuição marginal de cada novo bloco.


## 2. Contexto

No Q8 original (`03_bayes_hierarquico.ipynb` + `q8_bayes_hierarquico.py`) o grupo optou por excluir `key`, `mode` e `time_signature` da regressão por dois motivos:
1. Cardinalidade média-alta (`key` 12 níveis, `time_signature` 5 níveis) inflava dummies sem ganho estatístico claro.
2. Foco narrativo do Q8 era em como **features acústicas** (danceability, energy, etc.) variavam entre gêneros.

**O que muda agora (crítica do usuário / revisão):**
- O usuário questionou se remover essas variáveis não estaria descartando sinal real sobre hit potential.
- Decidimos trazê-las de volta com **encodings eficientes em parâmetros**:
  - `key` em cíclica (2 colunas) preserva a topologia circular do círculo de quintas.
  - `mode` permanece binário nativo.
  - `time_signature` em one-hot esparso (4/4 como referência).
- Também expandimos `genero_principal` (variável de agrupamento) para incluir sub-gêneros via **multi-hot top-30** — usando a maioria esmagadora das tracks (>80%) sem inflar 111 dummies.

**Eficiências aplicadas (não adicionamos features redundantes):**
- Não criamos `tonalidade` (24 níveis) nem `tonalidade_completa` (120 níveis) como features explícitas.
- A informação dessas combinações é absorvida por `(key_sin, key_cos) + mode + time_signature dummies` no preditor linear.
- Top-30 multi-hot descarta os 81 tokens de gênero raros (cada um com <0.5% das tracks) — puro ruído.


## 3. Hardware alvo e bibliotecas

- **GPU:** Google Colab T4 (16 GB VRAM, 2 560 CUDA cores, ~8 TFLOPS FP32).
- **PyMC:** 6.3.1 com backend NumPyro (`nuts_sampler='numpyro'`) para compilar logp+grad em XLA e rodar leapfrog em batch na GPU.
- **JAX:** backend CUDA 12.
- **ArviZ:** 1.3.0 para ELPD/LOO e plots.
- **Outros:** pandas, pyarrow, scikit-learn (z-score, MultiLabelBinarizer), h5py/h5netcdf (NetCDF).

Estimativas de tempo no T4 (full 90k tracks, K=46 features, G=111 gêneros):
| Modelo | Sampler | Wall time esperado |
|---|---|---|
| Gaussian (primário) | NUTS via NumPyro, 2 chains x 1000 draws | ~2-3 h |
| Bernoulli (primário) | ADVI 20k iter, mean-field | ~15-25 min |
| Drop-one (x6, K=45, 50k) | NUTS via NumPyro | ~35-60 min cada |

Total estimado: 6-10 h para o experimento completo (cabe em uma sessão Colab de 12 h).


In [ ]:
# C4 — Install dependencies
!pip install -q pymc==6.3.1 arviz==1.3.0 numpyro jax[cuda12] h5py h5netcdf pandas pyarrow scikit-learn scipy


In [ ]:
# C5 — Verify T4 GPU
!nvidia-smi


In [ ]:
# C6 — JAX GPU setup
import os
os.environ.setdefault('XLA_FLAGS', '--xla_force_host_platform_device_count=1')
os.environ.setdefault('JAX_PLATFORMS', 'cuda')

import jax
print('JAX devices:', jax.devices())
if jax.devices():
    jax.config.update('jax_default_device', jax.devices()[0])
    jax.config.update('jax_enable_x64', False)
print('Default device:', jax.default_device())


In [ ]:
# C7 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/spotify_challenge/insights-spotfy-grupo-4'
DATA_PARQUET = f'{PROJECT_ROOT}/data/processed/spotify_tracks_limpo.parquet'
RESULTS_DIR = f'{PROJECT_ROOT}/relatorio/analises/resultados'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('RESULTS_DIR:', RESULTS_DIR)


In [ ]:
# C8 — Load parquet
import pandas as pd

df = pd.read_parquet(DATA_PARQUET)
print(f'n faixas carregadas: {len(df):,}')
print('shape:', df.shape)
print('cols:', list(df.columns))
df.head(3)


In [ ]:
# C9 — Explore key/mode/time_signature/generos
for col in ['key', 'mode', 'time_signature']:
    print(f'\n--- {col} ---')
    print('dtype:', df[col].dtype)
    print('nulls:', df[col].isna().sum())
    print(df[col].value_counts(dropna=False).sort_index())

print('\n--- generos (string tokens) ---')
print('dtype:', df['generos'].dtype)
print('nulls:', df['generos'].isna().sum())
print(df['generos'].head(5).tolist())

print('\n--- genero_principal ---')
print('unique:', df['genero_principal'].nunique())
print(df['genero_principal'].value_counts().head(15))


## 10. Estratégias de encoding

| Feature | Encoding | Parâmetros | Justificativa |
|---|---|---|---|
| `key` (0..11) | cíclica: `key_sin = sin(2πk/12)`, `key_cos = cos(2πk/12)` | 2 contínuas | Preserva distância circular (C é adjacente a G, não a F#). Habilita modelo linear a aprender uma direção de fase 2D. |
| `mode` (0/1) | binário nativo | 1 | Já está em 0/1, sem transformação. |
| `time_signature` (3/4/5/6) | one-hot com 4/4 como referência | **3 dummies** (`ts_3, ts_5, ts_6`) | 4/4 domina (~90%); referência remove 1 dummy e mantém interpretabilidade. `ts_7` removido (7/8 é raro). |
| `generos` (111 tokens) | multi-hot top-30 em matriz esparsa CSR | 30 binárias | Tokens raros (<0.5% cada) só adicionariam ruído; csr_matrix com ~3% de densidade. |
| `tonalidade` (key+mode) | NÃO adicionada | 0 | Capturada implicitamente por `(key_sin, key_cos, mode)` + cross-terms. Adicionar 24 dummies inflaria 12x sem ganho. |
| `tonalidade_completa` (key+mode+ts) | NÃO adicionada | 0 | Capturada implicitamente por `(key_sin, key_cos, mode, ts_dummies)`. 120 dummies seria catastrófico. |

Total de features novas: **2 (key) + 1 (mode) + 3 (ts) + 30 (genero_top30) = 36**.  
Baseline: 10 features.  
**Total estendido: 46 colunas** (`K_ext = 46`).


In [ ]:
# C11 — Feature engineering
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix

# 11.1 — Ciclica para key
df['key_sin'] = np.sin(2 * np.pi * df['key'] / 12.0)
df['key_cos'] = np.cos(2 * np.pi * df['key'] / 12.0)

# 11.2 — Mode binario: ja vem 0/1, garantir int.
# Renomeado para mode_bin para evitar colisao de nome com a coluna 'mode' original.
df['mode_bin'] = df['mode'].astype(int)

# 11.3 — Time signature one-hot com 4/4 como referencia.
# Plan original: 4 dummies (ts_3, ts_5, ts_6, ts_7) -> K_ext = 47.
# REVISADO: como time_signature==7 (7/8) eh raro (<0.5% das tracks), removemos ts_7
# e mantemos 3 dummies -> K_ext = 46. Documentado no plano e em C14.
df['ts_3'] = (df['time_signature'] == 3).astype(int)
df['ts_5'] = (df['time_signature'] == 5).astype(int)
df['ts_6'] = (df['time_signature'] == 6).astype(int)
# ts_7 removido (raro): na maioria dos datasets modernos 7/8 eh <0.1%.

# NOTA: top-30 generos sera calculado em C13 sobre df_clean (apos cleaning),
# para evitar data leakage entre o top-30 e o df de treino.
print('Features harmonicas criadas (key_sin/key_cos/mode_bin + 3 dummies de ts)')


In [ ]:
# C12 — Clean data + z-score
NON_MUSICAL_GENRES = ['sleep', 'study', 'comedy', 'kids', 'children', 'new-age']

def is_non_music(s):
    if pd.isna(s):
        return False
    return any(g in NON_MUSICAL_GENRES for g in str(s).lower().split())

mask_non_music = df['generos'].apply(is_non_music)
df_clean = df[~mask_non_music].copy()
print(f'Removidas {mask_non_music.sum():,} faixas com genero nao-musical. Restam {len(df_clean):,}.')

# 12.1 — Z-score nas 10 features continuas originais + 2 key (key_sin/key_cos ja estao em [-1,1],
#         mas z-score garante unit-scale para o sampler).
BASELINE_FEATS = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'explicit',
]

df_clean['explicit'] = df_clean['explicit'].astype(int)

# Z-score continuo (excluindo explicit, que ja eh 0/1)
cont_feats = [f for f in BASELINE_FEATS if f != 'explicit']
means = df_clean[cont_feats].mean()
stds = df_clean[cont_feats].std()
df_clean[cont_feats] = (df_clean[cont_feats] - means) / stds

# key_sin/key_cos ja estao em [-1,1] mas z-score para uniformizar escala
for c in ['key_sin', 'key_cos']:
    mu, sd = df_clean[c].mean(), df_clean[c].std()
    df_clean[c] = (df_clean[c] - mu) / sd
    print(f'{c}: mu={mu:.3f}, sd={sd:.3f}')

print('Z-score aplicado em', cont_feats, '+ key_sin/key_cos.')


In [ ]:
# C13 — Build baseline X (10) e extended X (10 + 2 + 1 + 3 + 30 = 46)
from scipy.sparse import hstack as sparse_hstack, csr_matrix

# 13.1 — Baseline X: 10 features originais
df_clean = df_clean.dropna(subset=BASELINE_FEATS + ['key_sin', 'key_cos', 'mode_bin']
                            + ['ts_3', 'ts_5', 'ts_6']).reset_index(drop=True)
print(f'n apos dropna: {len(df_clean):,}')

X_base = df_clean[BASELINE_FEATS].values.astype('float32')
print('X_base shape:', X_base.shape)

# 13.2 — Extended X (sem genero_top30, parte densa)
NEW_FEATS = ['key_sin', 'key_cos', 'mode_bin', 'ts_3', 'ts_5', 'ts_6']
X_new_dense = df_clean[NEW_FEATS].values.astype('float32')
X_dense = np.concatenate([X_base, X_new_dense], axis=1)
print('X_dense shape (10 + 6):', X_dense.shape)

# 13.3 — genero_top30 multi-hot esparso, calculado SOBRE df_clean (sem data leakage).
tokens_clean = df_clean['generos'].fillna('').str.lower().str.split()
all_tokens_clean = pd.Series([t for toks in tokens_clean for t in toks])
TOP30_GENRES = all_tokens_clean.value_counts().head(30).index.tolist()
print('Top-30 tokens (sobre df_clean):', TOP30_GENRES[:10], '... (total', len(TOP30_GENRES), ')')

mlb = MultiLabelBinarizer(classes=TOP30_GENRES)
G_top30_clean = csr_matrix(mlb.fit_transform(tokens_clean).astype(np.float32))
print('G_top30_clean shape:', G_top30_clean.shape, '| nnz:', G_top30_clean.nnz,
      '| densidade:', G_top30_clean.nnz / (G_top30_clean.shape[0] * G_top30_clean.shape[1]))

# Cobertura cumulativa das tracks por estar em >=1 token do top-30
cobertura = (G_top30_clean.sum(axis=1) > 0).mean()
print(f'Cobertura por tracks: {cobertura:.2%}')

# Concatena denso + esparso para extended X completo
X_ext_dense_with_genre = sparse_hstack([csr_matrix(X_dense), G_top30_clean]).tocsr()
print('X_extended full shape (denso+esparso):', X_ext_dense_with_genre.shape)

# 13.4 — Genero idx para agrupamento hierarquico (111 niveis)
genero_cats = sorted(df_clean['genero_principal'].unique())
genero_to_idx = {g: i for i, g in enumerate(genero_cats)}
df_clean['genero_idx'] = df_clean['genero_principal'].map(genero_to_idx).astype('int32')
n_generos = len(genero_cats)
print(f'n_generos: {n_generos}')

# 13.5 — Targets (calculados uma vez; passados explicitamente para build_model).
y_pop = df_clean['popularity'].values.astype('float32')
mask_pos = y_pop > 0  # exclui popularity==0 da Gaussian
thresh = np.quantile(y_pop[mask_pos], 0.75)
y_top25 = (y_pop >= thresh).astype('int32')
print(f'thresh top25: {thresh:.1f} | pos rate: {y_top25.mean():.3f}')

# 13.6 — Feature names alinhados (K_ext = 10 + 6 + 30 = 46)
EXT_FEATURE_NAMES = BASELINE_FEATS + NEW_FEATS + TOP30_GENRES
assert len(EXT_FEATURE_NAMES) == 46, f'esperado 46, obtido {len(EXT_FEATURE_NAMES)}'
print('Total feature names:', len(EXT_FEATURE_NAMES))
print('Baseline features:', len(BASELINE_FEATS))


In [ ]:
# C14a — Smoke test (5 min): ADVI Bernoulli com 3 features (key_sin, key_cos, mode_bin)
#
# Decisao automatica:
#   GO    se alguma |mu_beta| > 0.05 E HDI 94% exclui 0 → seguir para C16+ (extended pipeline)
#   NO-GO se nenhuma atinge o limiar → pular extended fits, economizar 6-10h de T4
#
# Limiar 0.05: em escala padronizada, 0.05 sigma em logit ≈ 1.25% de mudança em P(hit).
#              Abaixo disso é ruído estatístico dado o tamanho da amostra (20k tracks).
# Trade-off: 5-10 min investidos aqui podem economizar 6-10h de fit sem ganho real.

import time
import pymc as pm
import arviz as az
import numpy as np
import pandas as pd

SMOKE_N = 20_000
SMOKE_FEATS = BASELINE_FEATS + ['key_sin', 'key_cos', 'mode_bin']  # 10 + 3 = 13 features
SMOKE_THRESHOLD = 0.05
NEW_FEATS_SMOKE = ['key_sin', 'key_cos', 'mode_bin']

# Subset deterministico (semente diferente do fit principal para nao interferir)
rng_smoke = np.random.default_rng(SEED + 100)
smoke_idx = rng_smoke.choice(len(df_clean), size=SMOKE_N, replace=False)
X_smoke = df_clean[SMOKE_FEATS].values[smoke_idx].astype('float32')
g_idx_smoke = df_clean['genero_idx'].values[smoke_idx].astype('int32')
y_top25_smoke = y_top25[smoke_idx]

print(f'[smoke] ADVI Bernoulli com {SMOKE_N:,} tracks, K={len(SMOKE_FEATS)} features')
print(f'[smoke] Features adicionadas: {NEW_FEATS_SMOKE}')

t0 = time.time()
with pm.Model(coords={'genero': genero_cats, 'feature': SMOKE_FEATS}) as smoke_model:
    X_data = pm.Data('X', X_smoke)
    g_data = pm.Data('genero_idx', g_idx_smoke)
    
    mu_alpha = pm.Normal('mu_alpha', mu=0.0, sigma=10.0)
    sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=10.0)
    mu_beta = pm.Normal('mu_beta', mu=0.0, sigma=2.5, dims='feature')
    sigma_beta = pm.HalfNormal('sigma_beta', sigma=2.5, dims='feature')
    
    z_alpha = pm.Normal('z_alpha', mu=0.0, sigma=1.0, dims='genero')
    z_beta = pm.Normal('z_beta', mu=0.0, sigma=1.0, dims=('genero', 'feature'))
    
    alpha_g = pm.Deterministic('alpha_g', mu_alpha + sigma_alpha * z_alpha, dims='genero')
    beta_g = pm.Deterministic('beta_g', mu_beta + sigma_beta * z_beta, dims=('genero', 'feature'))
    
    mu = alpha_g[g_data] + (X_data * beta_g[g_data]).sum(axis=1)
    pm.Bernoulli('y_obs', p=pm.math.sigmoid(mu), observed=y_top25_smoke)
    
    approx = pm.fit(n=10_000, method='advi', random_seed=SEED, progressbar=False,
                    obj_optimizer=pm.adam(learning_rate=5e-3))

idata_smoke = approx.sample(draws=1_000, random_seed=SEED)
elapsed = time.time() - t0
print(f'[smoke] Concluido em {elapsed/60:.1f} min')

# Extrair mu_beta + HDI para as 3 features novas
mu_beta_new = idata_smoke.posterior['mu_beta'].sel(feature=NEW_FEATS_SMOKE).mean(dim=('chain', 'draw')).values
hdi_new = az.hdi(idata_smoke.posterior['mu_beta'].sel(feature=NEW_FEATS_SMOKE), hdi_prob=0.94).values

# Decisao automatica
print('\n=== SMOKE TEST RESULT ===')
go_decision = False
for i, feat in enumerate(NEW_FEATS_SMOKE):
    sig = (hdi_new[i, 0] > 0) or (hdi_new[i, 1] < 0)
    mag = abs(mu_beta_new[i])
    if sig and mag > SMOKE_THRESHOLD:
        go_decision = True
    print(f'  {feat:12s}: mu_beta={mu_beta_new[i]:+.3f}  HDI=[{hdi_new[i, 0]:+.3f}, {hdi_new[i, 1]:+.3f}]  |mu|={mag:.3f}  sig={sig}')

decision_str = 'GO — seguir para C16+ (extended features valem o pipeline completo)' if go_decision else 'NO-GO — features nao agregam; pular extended fits para economizar 6-10h'
print(f'\n[DECISION] {decision_str}')

# Salvar para registro
smoke_df = pd.DataFrame({
    'feature': NEW_FEATS_SMOKE,
    'mu_beta': mu_beta_new,
    'hdi_lo': hdi_new[:, 0],
    'hdi_hi': hdi_new[:, 1],
    'abs_mu': np.abs(mu_beta_new),
    'significant': [(hdi_new[i, 0] > 0) or (hdi_new[i, 1] < 0) for i in range(3)],
})
smoke_df['decision'] = 'GO' if go_decision else 'NO-GO'
smoke_df.to_csv(f'{RESULTS_DIR}/q9_smoke_test_results.csv', index=False)
idata_smoke.to_netcdf(f'{RESULTS_DIR}/q9_smoke_test.nc', engine='h5netcdf')
print(f'[smoke] Salvos: q9_smoke_test_results.csv, q9_smoke_test.nc')


## 14. Modelo hierárquico — non-centered parameterization

Mantemos a mesma estrutura do Q8 (1 + features | genero), com **non-centered parameterization** para preservar geometria saudável do NUTS mesmo com K_ext = 46.

**Hiperpriori:**
- `mu_alpha ~ Normal(0, 10)`  — intercepto global.
- `sigma_alpha ~ HalfNormal(10)`  — SD entre gêneros para o intercepto.
- `mu_beta[k] ~ Normal(0, 2.5)` para k = 0..K-1  — efeito global por feature.
- `sigma_beta[k] ~ HalfNormal(2.5)` para k = 0..K-1  — SD entre gêneros por feature.

**Efeitos aleatórios (não-centrados):**
- `z_alpha[g] ~ Normal(0, 1)` para g = 0..110
- `z_beta[g, k] ~ Normal(0, 1)` para g = 0..110, k = 0..K-1

**Deterministics:**
- `alpha_g[g] = mu_alpha + sigma_alpha * z_alpha[g]`
- `beta_g[g, k] = mu_beta[k] + sigma_beta[k] * z_beta[g, k]`

**Preditor linear:**
- Gaussian: `mu_i = alpha_g[g_i] + sum_k beta_g[g_i, k] * X[i, k]`, `y_i ~ Normal(mu_i, sigma_y)`
- Bernoulli: `p_i = sigmoid(mu_i)`, `y_i ~ Bernoulli(p_i)`

**Parâmetros totais (extended, K=46, G=111):**
- `1 (mu_alpha) + 1 (sigma_alpha) + K=46 (mu_beta) + K=46 (sigma_beta) + G=111 (z_alpha) + G·K=5,106 (z_beta) = 5,311`
- Gaussian adiciona +1 (`sigma_y`) = **5,312 parâmetros latentes totais**.
- (Se mantivéssemos K=47 com ts_7, G·K=5,217, total=5,422; +1 sigma_y = 5,423.)

**Implementação:** matriz `X` envolvida em `pm.Data(...)` para permitir `pm.set_data()` ao trocar entre baseline/extended sem recompilar o grafo.


In [ ]:
# C15 — Build model definition (parametrizado por X, genero_idx, n_generos, n_features, family)
import pymc as pm
import numpy as np

def build_model(X, genero_idx, n_generos, n_features, family='gaussian',
                feature_names=None, y_pop=None, y_top25=None):
    """Constroi o modelo hierarquico nao-centrado.

    Parametros
    ----------
    X : array-like (N, K) ou scipy.sparse (N, K)
        Matriz de features. Pode ser densa ou esparsa.
    genero_idx : np.ndarray (N,) de int
        Indice inteiro do genero por faixa (0..G-1).
    n_generos : int
        Numero de grupos G.
    n_features : int
        Numero de features K.
    family : 'gaussian' | 'bernoulli'
        Familia do likelihood.
    feature_names : list[str] | None
        Nomes das features para coords (melhora plots/dims). Se None, usa
        `range(n_features)` como fallback numerico.
    y_pop : np.ndarray | None
        Targets Gaussian (popularidade continua). Obrigatorio se family=='gaussian'.
    y_top25 : np.ndarray | None
        Targets Bernoulli (top-25 binario). Obrigatorio se family=='bernoulli'.

    Notas
    -----
    - pm.Data aceita matrizes esparsas (scipy.sparse.csr_matrix), mas ao fazer
      `X_data * beta_g[g]` o JAX/NumPyro materializa o produto como denso na GPU.
      Para N=90k e K=46 isso eh ~16 MB denso por batch, gerenciavel.
    - y_pop/y_top25 sao passados explicitamente (sem globals()) — facilita testabilidade
      e elimina side-effects entre fits.
    """
    if feature_names is None:
        feature_names = [f'feat_{i}' for i in range(n_features)]
    assert len(feature_names) == n_features, \
        f'feature_names len={len(feature_names)} != n_features={n_features}'

    coords = {
        'genero': genero_cats,  # lista de strings (nomes dos generos)
        'feature': feature_names,  # lista de strings (nomes das features)
    }

    with pm.Model(coords=coords) as model:
        # pm.Data: permite trocar X via pm.set_data() sem recompilar.
        # Aceita sparse, mas o produto X*beta_g materializa como denso no GPU.
        X_data = pm.Data('X', X)

        # Hiperpriori
        mu_alpha = pm.Normal('mu_alpha', mu=0.0, sigma=10.0)
        sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=10.0)
        mu_beta = pm.Normal('mu_beta', mu=0.0, sigma=2.5, dims='feature')
        sigma_beta = pm.HalfNormal('sigma_beta', sigma=2.5, dims='feature')

        # Efeitos aleatorios nao-centrados
        z_alpha = pm.Normal('z_alpha', mu=0.0, sigma=1.0, dims='genero')
        z_beta = pm.Normal('z_beta', mu=0.0, sigma=1.0, dims=('genero', 'feature'))

        # Deterministics
        alpha_g = pm.Deterministic('alpha_g', mu_alpha + sigma_alpha * z_alpha, dims='genero')
        beta_g = pm.Deterministic('beta_g', mu_beta + sigma_beta * z_beta, dims=('genero', 'feature'))

        # Preditor linear: alpha_g[g_i] + sum_k beta_g[g_i, k] * X[i, k]
        g = pm.Data('genero_idx', genero_idx)
        mu = alpha_g[g] + (X_data * beta_g[g]).sum(axis=1)

        if family == 'gaussian':
            assert y_pop is not None, 'build_model(gaussian) requer y_pop'
            sigma_y = pm.HalfNormal('sigma_y', sigma=20.0)
            pm.Normal('y_obs', mu=mu, sigma=sigma_y, observed=y_pop)
        elif family == 'bernoulli':
            assert y_top25 is not None, 'build_model(bernoulli) requer y_top25'
            pm.Bernoulli('y_obs', p=pm.math.sigmoid(mu), observed=y_top25)
        else:
            raise ValueError(family)

    return model

print('build_model definido (versao com y_pop/y_top25 explicitos e feature_names)')


In [ ]:
# C16 — fit_baseline_bernoulli (ADVI 20k em FULL data — Bernoulli eh rapido com ADVI)
import time
import pymc as pm

SEED = 42
# SUBSAMPLE_N agora eh usado APENAS no drop-one (C23) para velocidade.
# Bernoulli em ADVI cabe nos 90k completos sem problema.
SUBSAMPLE_N = 50_000

X_base_full = X_base
g_idx_full = df_clean['genero_idx'].values.astype('int32')
y_pop_full = y_pop
y_top25_full = y_top25

print(f'[bernoulli:baseline] ADVI 20k em FULL data n={len(g_idx_full):,}, K={X_base_full.shape[1]}...')
t0 = time.time()
with build_model(X_base_full, g_idx_full, n_generos, X_base_full.shape[1],
                family='bernoulli',
                feature_names=BASELINE_FEATS,
                y_top25=y_top25_full) as _:
    approx_base_b = pm.fit(n=20_000, method='advi', random_seed=SEED,
                            progressbar=False,
                            obj_optimizer=pm.adam(learning_rate=5e-3))
idata_base_b = approx_base_b.sample(draws=2_000, random_seed=SEED)
elapsed = time.time() - t0
print(f'[bernoulli:baseline] concluido em {elapsed/60:.1f} min')

idata_base_b.to_netcdf(f'{RESULTS_DIR}/q9_baseline_bernoulli.nc', engine='h5netcdf')
print('salvo: q9_baseline_bernoulli.nc')


In [ ]:
# C17 — fit_extended_bernoulli (FULL data, X estendido com genero_multi esparso)
X_ext_full = X_ext_dense_with_genre
print(f'[bernoulli:extended] ADVI 20k em FULL data n={len(g_idx_full):,}, K={X_ext_full.shape[1]}...')
t0 = time.time()
with build_model(X_ext_full, g_idx_full, n_generos, X_ext_full.shape[1],
                family='bernoulli',
                feature_names=EXT_FEATURE_NAMES,
                y_top25=y_top25_full) as _:
    approx_ext_b = pm.fit(n=20_000, method='advi', random_seed=SEED,
                            progressbar=False,
                            obj_optimizer=pm.adam(learning_rate=5e-3))
idata_ext_b = approx_ext_b.sample(draws=2_000, random_seed=SEED)
elapsed = time.time() - t0
print(f'[bernoulli:extended] concluido em {elapsed/60:.1f} min')

idata_ext_b.to_netcdf(f'{RESULTS_DIR}/q9_extended_bernoulli.nc', engine='h5netcdf')
print('salvo: q9_extended_bernoulli.nc')


In [ ]:
# C18 — fit_baseline_gaussian (NUTS via NumPyro em FULL data, 90k, target_accept=0.95)
X_base_pos = X_base[mask_pos]
g_idx_pos = df_clean['genero_idx'].values[mask_pos].astype('int32')
y_pop_pos = y_pop[mask_pos]
print(f'[gaussian:baseline] NUTS via NumPyro, full data n={len(g_idx_pos):,}, K={X_base_pos.shape[1]}...')

t0 = time.time()
with build_model(X_base_pos, g_idx_pos, n_generos, X_base_pos.shape[1],
                family='gaussian',
                feature_names=BASELINE_FEATS,
                y_pop=y_pop_pos) as _:
    idata_base_g = pm.sample(
        draws=1000, tune=1000, chains=2,
        nuts_sampler='numpyro', target_accept=0.95,  # 0.95 para fits Gaussian primarios
        random_seed=SEED, progressbar=False,
    )
elapsed = time.time() - t0
print(f'[gaussian:baseline] NUTS concluido em {elapsed/60:.1f} min')

idata_base_g.to_netcdf(f'{RESULTS_DIR}/q9_baseline_gaussian.nc', engine='h5netcdf')
print('salvo: q9_baseline_gaussian.nc')


In [ ]:
# C19 — fit_extended_gaussian (NUTS via NumPyro, full data, K=46, target_accept=0.95)
X_ext_pos = X_ext_dense_with_genre[mask_pos]
print(f'[gaussian:extended] NUTS via NumPyro, full data, K={X_ext_pos.shape[1]}...')

t0 = time.time()
with build_model(X_ext_pos, g_idx_pos, n_generos, X_ext_pos.shape[1],
                family='gaussian',
                feature_names=EXT_FEATURE_NAMES,
                y_pop=y_pop_pos) as _:
    idata_ext_g = pm.sample(
        draws=1000, tune=1000, chains=2,
        nuts_sampler='numpyro', target_accept=0.95,  # 0.95 para fits Gaussian primarios
        random_seed=SEED, progressbar=False,
    )
elapsed = time.time() - t0
print(f'[gaussian:extended] NUTS concluido em {elapsed/60:.1f} min')

idata_ext_g.to_netcdf(f'{RESULTS_DIR}/q9_extended_gaussian.nc', engine='h5netcdf')
print('salvo: q9_extended_gaussian.nc')


## 20. Análise de impacto — comparar μ_β (efeito global)

Pergunta: ao adicionar 37 novas features, **os 10 coeficientes originais mudam materialmente**?
- Se sim: as novas features estavam absorvendo parte do efeito (multicolinearidade).
- Se não: as novas features trazem sinal ortogonal (bom).

Também: rankeamos as 47 features por impacto global e identificamos as 10 com maior |μ_β|.


In [ ]:
# C21 — Comparar mu_beta: baseline vs extended
import arviz as az
import numpy as np
import pandas as pd

def posterior_mean_hdi(idata, var, hdi_prob=0.94):
    d = idata.posterior[var]
    mean = d.mean(dim=('chain', 'draw')).values
    hdi = az.hdi(d, hdi_prob=hdi_prob).values
    if hdi.ndim == 1:
        lo, hi = hdi[0], hdi[1]
    else:
        lo, hi = hdi[..., 0], hdi[..., 1]
    return mean, lo, hi

# Coefs baseline (10)
mb_base, lo_base, hi_base = posterior_mean_hdi(idata_base_g, 'mu_beta')

# Coefs extended (47) — pegamos so os 10 primeiros (mesma ordem das BASELINE_FEATS)
mb_ext, lo_ext, hi_ext = posterior_mean_hdi(idata_ext_g, 'mu_beta')
mb_ext_base10 = mb_ext[:len(BASELINE_FEATS)]
lo_ext_base10 = lo_ext[:len(BASELINE_FEATS)]
hi_ext_base10 = hi_ext[:len(BASELINE_FEATS)]

comp = pd.DataFrame({
    'feature': BASELINE_FEATS,
    'mu_beta_baseline': mb_base,
    'mu_beta_extended': mb_ext_base10,
    'abs_delta': np.abs(mb_ext_base10 - mb_base),
})
comp = comp.sort_values('abs_delta', ascending=False)
print(comp.to_string(index=False))

# Ranking geral das 47 features extended
rank_ext = pd.DataFrame({
    'feature_idx': np.arange(len(mb_ext)),
    'feature_name': EXT_FEATURE_NAMES,
    'mu_beta': mb_ext,
    'abs_mu_beta': np.abs(mb_ext),
    'hdi_lo': lo_ext,
    'hdi_hi': hi_ext,
})
rank_ext = rank_ext.sort_values('abs_mu_beta', ascending=False).reset_index(drop=True)
print('\nTop-10 mais impactuais (|mu_beta|):')
print(rank_ext.head(10)[['feature_name', 'mu_beta', 'hdi_lo', 'hdi_hi']].to_string(index=False))


In [ ]:
# C22a — LOO / WAIC comparison (baseline vs extended)
import arviz as az
import pandas as pd

print('=== LOO comparison ===')
try:
    az.loo(idata_base_g, pointwise=True)
    az.loo(idata_ext_g, pointwise=True)
    comp = az.compare({'baseline': idata_base_g, 'extended': idata_ext_g})
    print(comp)
    comp.to_csv(f'{RESULTS_DIR}/q9_loo_comparison.csv')
    print('salvo: q9_loo_comparison.csv')
except Exception as e:
    print(f'LOO falhou (provavelmente log_likelihood nao computado): {e}')
    print('Para habilitar LOO, adicione idata.extend(pm.compute_log_likelihood(idata)) apos pm.sample().')

print('\n=== WAIC comparison (fallback) ===')
try:
    waic_base = az.waic(idata_base_g)
    waic_ext = az.waic(idata_ext_g)
    print('WAIC baseline:', waic_base)
    print('WAIC extended:', waic_ext)
    waic_df = pd.DataFrame({
        'model': ['baseline', 'extended'],
        'elpd_waic': [waic_base.elpd_waic, waic_ext.elpd_waic],
        'se_elpd_waic': [waic_base.se, waic_ext.se],
        'p_waic': [waic_base.p_waic, waic_ext.p_waic],
    })
    waic_df.to_csv(f'{RESULTS_DIR}/q9_waic_comparison.csv', index=False)
    print('salvo: q9_waic_comparison.csv')
except Exception as e:
    print(f'WAIC falhou: {e}')


## 22b. PPC — goodness-of-fit (in-sample) via sample_posterior_predictive

**PPC eh goodness-of-fit (in-sample) usando `pm.sample_posterior_predictive`. Out-of-sample comparison vem do LOO em C22a.** Isso evita 2x tempo de fit por holdout refit (cada fit NUTS Gaussian no T4 leva ~2-3 h; refit em 80k subset adicionaria ~2 h sem ganho real).

**Por que NAO fazemos holdout refit?**
1. PSIS-LOO via `az.loo()` ja eh exato para o modelo (Pareto-smoothed importance sampling leave-one-out cross-validation), sem variancia Monte Carlo adicional.
2. Refit de ambos os modelos Gaussian (baseline + extended) em 80k subset custaria ~2 h extras de T4 GPU, sem mudar a conclusao (LOO ja captura isso).
3. PPC in-sample ainda da diagnostico util: calibracao, residuos, cobertura do HDI.

**Implementacao:** para cada `idata` ja ajustado (C16-C19), rebuild do modelo com os mesmos `pm.Data('X', X_train)` + `pm.Data('genero_idx', g_idx_train)`, e chamada de `pm.sample_posterior_predictive(idata, var_names=['y_obs'], predictions=True, extend_inferencedata=True, random_seed=SEED)`. Sem recompilacao do grafo longo.

**Metricas calculadas (in-sample, em TRAINING set):**
- Gaussian: RMSE, MAE da media preditiva vs `y_pop` observado.
- Bernoulli: Brier score, log-loss, accuracy@0.5 da `p_pred_mean` vs `y_top25` observado.

**IMPORTANTE:** Estas metricas NAO sao generalizacao out-of-sample. Para comparar baseline vs extended out-of-sample, ver `q9_loo_comparison.csv` (C22a).

In [ ]:
# C22b — PPC in-sample via pm.sample_posterior_predictive (sem holdout refit)
#
# DECISAO: NAO split holdout / refit. Out-of-sample vem do LOO (C22a).
# Aqui computamos goodness-of-fit no TRAINING set usando a posterior completa
# (nao apenas a media). Metricas: RMSE/MAE (Gaussian), Brier/log-loss/accuracy (Bernoulli).
import numpy as np
import pandas as pd
import pymc as pm

print('[PPC] sample_posterior_predictive para 4 modelos (in-sample)...')

def ppc_in_sample(idata, X, g_idx, y, family, label, feature_names):
    """Roda pm.sample_posterior_predictive no TRAINING set e computa metricas.

    Nao ha recompilacao do grafo longo (PyMC infere a estrutura a partir do idata).
    Apenas pm.Data com os mesmos X/g_idx sao configurados.
    """
    n_features = X.shape[1]
    n_obs = len(y)

    if family == 'gaussian':
        with build_model(X, g_idx, n_generos, n_features,
                         family='gaussian',
                         feature_names=feature_names,
                         y_pop=y):
            ppc = pm.sample_posterior_predictive(
                idata, var_names=['y_obs'],
                predictions=True,
                extend_inferencedata=True,
                random_seed=SEED,
                progressbar=False,
            )
        # ppc.predictions['y_obs']: shape (chain, draw, N)
        y_pred = ppc.predictions['y_obs'].values
        y_pred_mean = y_pred.mean(axis=(0, 1))
        rmse = float(np.sqrt(np.mean((y_pred_mean - y) ** 2)))
        mae = float(np.mean(np.abs(y_pred_mean - y)))
        return {
            'model': label,
            'family': 'gaussian',
            'n_train': n_obs,
            'rmse_in_sample': rmse,
            'mae_in_sample': mae,
        }
    elif family == 'bernoulli':
        with build_model(X, g_idx, n_generos, n_features,
                         family='bernoulli',
                         feature_names=feature_names,
                         y_top25=y):
            ppc = pm.sample_posterior_predictive(
                idata, var_names=['y_obs'],
                predictions=True,
                extend_inferencedata=True,
                random_seed=SEED,
                progressbar=False,
            )
        y_pred = ppc.predictions['y_obs'].values
        p_pred_mean = y_pred.mean(axis=(0, 1))
        p_pred_clipped = np.clip(p_pred_mean, 1e-7, 1 - 1e-7)
        brier = float(np.mean((p_pred_mean - y) ** 2))
        logloss = float(-np.mean(
            y * np.log(p_pred_clipped) + (1 - y) * np.log(1 - p_pred_clipped)
        ))
        acc = float(((p_pred_mean > 0.5).astype(int) == y).mean())
        return {
            'model': label,
            'family': 'bernoulli',
            'n_train': n_obs,
            'brier_in_sample': brier,
            'log_loss_in_sample': logloss,
            'accuracy_in_sample': acc,
        }

ppc_rows = []

# Gaussian baseline — treinado em X_base_pos (subset popularity>0), K=10
print('\n[PPC] Gaussian baseline (K=10, n={:,})...'.format(len(g_idx_pos)))
ppc_rows.append(ppc_in_sample(
    idata_base_g, X_base_pos, g_idx_pos, y_pop_pos,
    'gaussian', 'gaussian_baseline', BASELINE_FEATS
))

# Gaussian extended — treinado em X_ext_pos, K=46
print('[PPC] Gaussian extended (K=46, n={:,})...'.format(len(g_idx_pos)))
ppc_rows.append(ppc_in_sample(
    idata_ext_g, X_ext_pos, g_idx_pos, y_pop_pos,
    'gaussian', 'gaussian_extended', EXT_FEATURE_NAMES
))

# Bernoulli baseline — treinado em X_base_full, K=10
print('[PPC] Bernoulli baseline (K=10, n={:,})...'.format(len(g_idx_full)))
ppc_rows.append(ppc_in_sample(
    idata_base_b, X_base_full, g_idx_full, y_top25_full,
    'bernoulli', 'bernoulli_baseline', BASELINE_FEATS
))

# Bernoulli extended — treinado em X_ext_full, K=46
print('[PPC] Bernoulli extended (K=46, n={:,})...'.format(len(g_idx_full)))
ppc_rows.append(ppc_in_sample(
    idata_ext_b, X_ext_full, g_idx_full, y_top25_full,
    'bernoulli', 'bernoulli_extended', EXT_FEATURE_NAMES
))

ppc_df = pd.DataFrame(ppc_rows)
print('\n=== PPC results (in-sample goodness-of-fit) ===')
print(ppc_df.to_string(index=False))
print('\nNOTA: Metricas in-sample. Out-of-sample = LOO em C22a (q9_loo_comparison.csv).')
ppc_df.to_csv(f'{RESULTS_DIR}/q9_ppc_results.csv', index=False)
print('salvo: q9_ppc_results.csv')

## 22. Sensitivity — drop-one feature analysis

Para quantificar a **contribuição marginal** de cada novo bloco, removemos sistematicamente cada um e re-ajustamos a Gaussian no subset de 50k:

1. **NO_KEY** — remove `key_sin`, `key_cos`.
2. **NO_MODE** — remove `mode_bin`.
3. **NO_TIME_SIG** — remove `ts_3`, `ts_5`, `ts_6`, `ts_7`.
4. **NO_GENRE_MULTI** — remove os 30 `genero_top30`.
5. **NO_TONALIDADE_INTERACTIONS** — remove cross-terms `key_sin*mode_bin`, `key_cos*mode_bin` (mantém apenas aditivos).
6. **NO_BASELINE_LEADER** — leave-one-out dentro das 10 originais (10 mini-fits, cada um remove uma feature).

Cada variante tem K = 47 - k (k = tamanho do bloco removido). Comparamos via **Δ μ_β nas features remanescentes** + Δ **LOO** quando disponível.


In [ ]:
# C23 — Drop-one loop (Gaussian NUTS via NumPyro em subsample 50k, target_accept=0.95)
#
# Variantes dropadas (K_ext = 46 base):
#   - NO_KEY             -> remove key_sin, key_cos (K=44)
#   - NO_MODE            -> remove mode_bin (K=45)
#   - NO_TIME_SIG        -> remove ts_3, ts_5, ts_6 (K=43)
#   - NO_GENRE_MULTI     -> remove os 30 genero_top30 (K=16)
#   - LOO baseline (10x) -> remove cada uma das 10 originais (K=9)
#
# NO_TONALIDADE_INTERACTIONS foi removido: o preditor linear eh puramente aditivo
# (sem cross-terms), entao nao ha o que remover. Capturamos essa informacao na
# comparacao direta baseline (K=10) vs extended sem generos (K=16).
#
# NOTA sobre pm.set_data / refactor: como n_features muda entre variantes,
# seria necessario um modelo NOVO por variante (X com K diferente nao cabe no mesmo
# grafo compilado). Workaround ideal seria fixar K=46 e maskar features com 0,
# mas isso polui o sampling. Aqui aceitamos o overhead de ~30-60s de compilacao
# por variante (tradeoff entre velocidade e clareza).
drop_groups = {
    'NO_KEY': ['key_sin', 'key_cos'],
    'NO_MODE': ['mode_bin'],
    'NO_TIME_SIG': ['ts_3', 'ts_5', 'ts_6'],
    'NO_GENRE_MULTI': TOP30_GENRES,
}

rng = np.random.default_rng(SEED)
sub_idx = rng.choice(len(df_clean), size=min(SUBSAMPLE_N, len(df_clean)), replace=False)

g_idx_sub = df_clean['genero_idx'].values[sub_idx].astype('int32')
y_pop_sub = y_pop[sub_idx]

drop_results = {}

# Indice das colunas densas (BASELINE_FEATS + NEW_FEATS) em X_dense
EXT_FEATS_LIST = BASELINE_FEATS + NEW_FEATS  # nomes das colunas densas

for group_name, feats_to_remove in drop_groups.items():
    # Para NO_KEY/NO_MODE/NO_TIME_SIG: sempre comecar de X_ext_dense_with_genre (full)
# e dropar APENAS as colunas densas do grupo. Mantemos TODOS os genero_top30.
    if group_name == 'NO_GENRE_MULTI':
        # Caso especial: remove os 30 genero_multi. Usa apenas X_dense (sem genero).
        keep_dense_idx = list(range(X_dense.shape[1]))  # todas as 16 densas
        X_new = X_dense[sub_idx][:, keep_dense_idx]
        feat_names = EXT_FEATS_LIST
    else:
        # Demais casos: remove colunas densas do grupo, MANTEM genero_multi.
        keep_dense_idx = [i for i, n in enumerate(EXT_FEATS_LIST) if n not in feats_to_remove]
        X_dense_sub = X_dense[sub_idx][:, keep_dense_idx]
        X_new = sparse_hstack([csr_matrix(X_dense_sub), G_top30_clean[sub_idx]]).tocsr()
        feat_names = [n for n in EXT_FEATS_LIST if n not in feats_to_remove] + TOP30_GENRES

    K_drop = X_new.shape[1]
    print(f'\n[drop-one:{group_name}] K={K_drop}...')

    t0 = time.time()
    with build_model(X_new, g_idx_sub, n_generos, K_drop,
                    family='gaussian',
                    feature_names=feat_names,
                    y_pop=y_pop_sub) as _:
        idata_drop = pm.sample(
            draws=750, tune=750, chains=2,
            nuts_sampler='numpyro', target_accept=0.95,  # 0.95 para accuracy
            random_seed=SEED, progressbar=False,
        )
    elapsed = time.time() - t0

    mb_drop, _, _ = posterior_mean_hdi(idata_drop, 'mu_beta')
    drop_results[group_name] = {
        'K': K_drop,
        'wall_time_min': elapsed / 60,
        'mu_beta_first10': mb_drop[:10].tolist(),
    }
    idata_drop.to_netcdf(f'{RESULTS_DIR}/q9_dropone_{group_name}.nc', engine='h5netcdf')
    print(f'  -> {elapsed/60:.1f} min, salvo q9_dropone_{group_name}.nc')

# Bonus: 10 mini-fits leave-one-out dentro das 10 originais
loo_results = {}
for feat_remove in BASELINE_FEATS:
    keep_idx = [i for i, n in enumerate(BASELINE_FEATS) if n != feat_remove]
    X_loo = X_base[sub_idx][:, keep_idx]
    feat_names_loo = [n for n in BASELINE_FEATS if n != feat_remove]
    K_loo = X_loo.shape[1]
    print(f'[LOO-baseline:{feat_remove}] K={K_loo}...')
    t0 = time.time()
    with build_model(X_loo, g_idx_sub, n_generos, K_loo,
                    family='gaussian',
                    feature_names=feat_names_loo,
                    y_pop=y_pop_sub) as _:
        idata_loo = pm.sample(
            draws=500, tune=500, chains=2,
            nuts_sampler='numpyro', target_accept=0.95,
            random_seed=SEED, progressbar=False,
        )
    elapsed = time.time() - t0
    loo_results[feat_remove] = {
        'K': K_loo,
        'wall_time_min': elapsed / 60,
    }
    idata_loo.to_netcdf(f'{RESULTS_DIR}/q9_dropone_baseline_{feat_remove}.nc', engine='h5netcdf')

print('\n[drop-one completo]')


## 24. Resultados globais

Ordenamos as features estendidas por impacto (`|μ_β|`), comparamos com a baseline e listamos as top-10 mais impactuais (incluindo as 10 originais + 37 novas).

Também mostramos como a variação entre gêneros (`σ_β`) se redistribui quando adicionamos as novas features — se uma feature original (ex: `danceability`) tinha `σ_β` alto e agora tem `σ_β` menor, é porque parte da variabilidade entre gêneros foi absorvida por `key_sin/key_cos/mode/ts/genero_top30`.


In [ ]:
# C25 — Top-10 features por impacto + variacao entre generos
import arviz as az
import pandas as pd
import numpy as np

# Top-10 mais impactuais
rank_ext_top = rank_ext.head(10).copy()
print('=== TOP-10 features estendidas por |mu_beta| ===')
print(rank_ext_top[['feature_name', 'mu_beta', 'hdi_lo', 'hdi_hi']].to_string(index=False))

# Comparacao de sigma_beta: baseline (10) vs extended (47)
sb_base, _, _ = posterior_mean_hdi(idata_base_g, 'sigma_beta')
sb_ext, _, _ = posterior_mean_hdi(idata_ext_g, 'sigma_beta')
sigma_comp = pd.DataFrame({
    'feature': BASELINE_FEATS,
    'sigma_beta_baseline': sb_base,
    'sigma_beta_extended': sb_ext[:len(BASELINE_FEATS)],
    'abs_delta_sigma': np.abs(sb_ext[:len(BASELINE_FEATS)] - sb_base),
})
sigma_comp = sigma_comp.sort_values('abs_delta_sigma', ascending=False)
print('\n=== Variacao entre generos (sigma_beta): baseline vs extended ===')
print(sigma_comp.to_string(index=False))

# Top-10 novas features com maior sigma_beta (entre-generos) — sao as que MAIS diferenciam generos
sigma_new = pd.DataFrame({
    'feature_name': EXT_FEATURE_NAMES,
    'sigma_beta': sb_ext,
}).iloc[len(BASELINE_FEATS):].sort_values('sigma_beta', ascending=False).head(10)
print('\n=== TOP-10 NOVAS features com maior sigma_beta (variacao entre generos) ===')
print(sigma_new.to_string(index=False))


In [ ]:
# C26 — Forest plot para top-10 features estendidas
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def manual_forest(idata, feature_idx, ax, title):
    d = idata.posterior['beta_g'].isel(feature=feature_idx)
    means = d.mean(dim=('chain', 'draw')).values
    lo = d.quantile(0.03, dim=('chain', 'draw')).values
    hi = d.quantile(0.97, dim=('chain', 'draw')).values
    n = len(means)
    y = np.arange(n)[::-1]
    sig = (lo > 0) | (hi < 0)
    for yi, m, l, h, s in zip(y, means, lo, hi, sig):
        c = '#1db954' if s else '#7a7166'
        ax.hlines(yi, l, h, color=c, linewidth=1.2, alpha=0.7)
        ax.plot(m, yi, 'o', color=c, markersize=3)
    ax.axvline(0, color='#c44b3e', linewidth=0.7, linestyle='--', alpha=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels(genero_cats, fontsize=7)
    ax.set_xlabel('efeito padronizado (CI 94%)')
    ax.set_title(title, fontsize=10)
    ax.grid(axis='x', linestyle=':', alpha=0.3)
    ax.text(0.99, 0.01, f'{int(sig.sum())}/{n} generos com CI fora de zero',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8)

for rank_i in range(10):
    feat_name = rank_ext_top.iloc[rank_i]['feature_name']
    feat_idx = rank_ext_top.iloc[rank_i]['feature_idx']
    fig, ax = plt.subplots(figsize=(10, max(8, 0.22 * n_generos)))
    manual_forest(idata_ext_g, feat_idx, ax,
                  title=f'Top-{rank_i+1}: {feat_name}  (mu_beta={rank_ext_top.iloc[rank_i]["mu_beta"]:.3f})')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/q9_forest_extended_{feat_name}.png', dpi=110)
    plt.close(fig)

print('10 forest plots salvos em', RESULTS_DIR)


## 27. Conclusões

Após o fit, preencha este resumo com base nos resultados reais:

- **Vale a pena incluir `key` (cíclica)?** Olhar Δ_μ_β em `key_sin`/`key_cos` vs o HDI dos outros coeficientes; comparar σ_β entre baseline e extended. Se σ_β das originais caiu, é porque key estava confundindo parte da variação entre gêneros.
- **Mode (major/minor) agrega?** Olhar magnitude do |μ_β| de `mode_bin` e quantos gêneros têm CI fora de zero no forest plot.
- **`time_signature` é ruído?** Se todos os 4 dummies (ts_3, ts_5, ts_6, ts_7) tiverem |μ_β| < 0.1 e HDI cruzando zero, é candidato a remoção.
- **`genero_top30` ajuda?** Ver Δ_μ_β nas features originais (especialmente em `danceability` e `valence`) entre baseline e extended. Queda grande = o multi-hot estava capturando algo que o genero_principal grouping já resumia.
- **Qual feature mais contribui?** O ranking top-10 em |μ_β| responde direto; o drop-one ranking (qual remoção mais deteriora LOO) responde a versão preditiva.

Recomendação: se `ts_*` sair não-significant, remova do modelo final. Se `key_sin/key_cos` sairem significativos, mantenha (e justifique o custo de 2 parâmetros).


In [ ]:
# C28 — Save CSVs e resumo final
import pandas as pd
import os

# CSV 1: comparacao global baseline vs extended
comp.to_csv(f'{RESULTS_DIR}/q9_global_effects_baseline_vs_extended.csv', index=False)

# CSV 2: ranking das 47 features extended
rank_ext.to_csv(f'{RESULTS_DIR}/q9_global_effects_extended.csv', index=False)

# CSV 3: sigma_beta baseline vs extended
sigma_comp.to_csv(f'{RESULTS_DIR}/q9_sigma_beta_baseline_vs_extended.csv', index=False)

# CSV 4: drop-one results
rows = []
for g, info in drop_results.items():
    rows.append({
        'group': g,
        'K': info['K'],
        'wall_time_min': info['wall_time_min'],
    })
for f, info in loo_results.items():
    rows.append({
        'group': f'LOO_BASELINE_{f}',
        'K': info['K'],
        'wall_time_min': info['wall_time_min'],
    })
pd.DataFrame(rows).to_csv(f'{RESULTS_DIR}/q9_dropone_results.csv', index=False)

# TXT: resumo
lines = []
lines.append('=== Q9 — Q8 v2 (features extras) ===')
lines.append(f'n faixas apos cleaning: {len(df_clean):,}')
lines.append(f'n generos (G): {n_generos}')
lines.append(f'K baseline: {len(BASELINE_FEATS)}')
lines.append(f'K extended: {len(EXT_FEATURE_NAMES)}')
lines.append('\n--- Top-10 features extended por |mu_beta| ---')
lines.append(rank_ext_top[['feature_name', 'mu_beta', 'hdi_lo', 'hdi_hi']].to_string(index=False))
lines.append('\n--- Sigma_beta baseline vs extended (10 originais) ---')
lines.append(sigma_comp.to_string(index=False))

with open(f'{RESULTS_DIR}/q9_resumo.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('Artefatos salvos em', RESULTS_DIR)
for fn in sorted(os.listdir(RESULTS_DIR)):
    if fn.startswith('q9_'):
        print(' ', fn)
